In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Fri May  1 02:07:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install textworld peft transformers bitsandbytes accelerate trl datasets pyyaml wandb -q
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!pip install -e LLaMA-Factory[metrics,bitsandbytes] -q
print("Done. RESTART RUNTIME NOW.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.2 MB/s eta 0:00:00
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 647, done.
remote: Counting objects: 100% (647/647), done.
remote: Compressing objects: 100% (489/489), done.
remote: Total 647 (delta 151), reused 403 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (647/647), 5.29 MiB | 21.75 MiB/s, done.
Resolving deltas: 100% (151/151), done.
  Installing build dependencies ... done
  Checki

In [ ]:
import textworld
import textworld.generator as gen
from tqdm import tqdm
import json, os

TEST_GAMES_DIR = "./games/test"
os.makedirs(TEST_GAMES_DIR, exist_ok=True)

print("Regenerating 100 test games (fixed seeds — identical to original)...")
test_game_paths = []

for game_id in tqdm(range(100)):
    options = textworld.GameOptions()
    options.seeds        = 10000 + game_id  # same seeds as original
    options.nb_rooms     = 3
    options.nb_objects   = 3
    options.quest_length = 5
    options.path         = TEST_GAMES_DIR

    try:
        game_file = gen.compile_game(gen.make_game(options))
        test_game_paths.append(game_file)
    except Exception:
        continue

with open('./data/test_game_paths.json', 'w') as f:
    json.dump(test_game_paths, f, indent=2)

print(f"Done. {len(test_game_paths)} test games regenerated.")
print("These are byte-for-byte identical to your original test games.")

Regenerating 100 test games (fixed seeds — identical to original)...


100%|██████████| 100/100 [01:43<00:00,  1.04s/it]

Done. 100 test games regenerated.
These are byte-for-byte identical to your original test games.


In [ ]:
# Test SFT model ALONE before any GRPO
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
base_only = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True
)
sft_model = PeftModel.from_pretrained(base_only, SFT_CKPT)
sft_model.eval()

test_obs = (
    "You are playing a text-based game. "
    "Read the observation carefully and choose the best action.\n\n"
    "Observation:\n-= Kitchen =-\nYou are in a kitchen.\n\n"
    "Valid actions: go north, go east, take key\n\n"
    "What is the best action to take? Reply with only the action."
)
msgs   = [{"role": "user", "content": test_obs}]
text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(sft_model.device)

for i in range(5):
    with torch.no_grad():
        out = sft_model.generate(
            **inputs, max_new_tokens=15,
            do_sample=True, temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    print(f"{i+1}. '{response}'")

del sft_model, base_only
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

1. 'take key'
2. 'take key'
3. 'take key'
4. 'take key'
5. 'take key'


In [ ]:
import shutil, os, json

# ── Restore checkpoints ──
for method in ['sft_qwen_textworld', 'ppo_qwen_textworld', 'grpo_qwen_textworld']:
    src = f'./restored/checkpoints/{method}'
    dst = f'./checkpoints/{method}'
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Restored: {method}")
    else:
        print(f"Missing: {method}")

# ── Restore data ──
os.makedirs('./data', exist_ok=True)
os.makedirs('./results', exist_ok=True)

# sft_data.json lives directly in restored/ not restored/data/
data_sources = {
    './data/sft_data.json':        './restored/sft_data.json',
    './data/test_game_paths.json': './restored/test_game_paths.json',
}
for dst, src in data_sources.items():
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Restored: {dst}")
    else:
        print(f"Missing in zip: {src} — will need to regenerate")

# ── Restore results ──
if os.path.exists('./restored/results'):
    shutil.copytree('./restored/results', './results', dirs_exist_ok=True)
    print("Restored: results/")

# ── Restore LlamaFactory dataset ──
os.makedirs('./LLaMA-Factory/data', exist_ok=True)
if os.path.exists('./data/sft_data.json'):
    shutil.copy('./data/sft_data.json', './LLaMA-Factory/data/textworld_sft.json')
    print("Restored: LlamaFactory dataset")

dataset_info = {
    "textworld_sft": {
        "file_name":  "textworld_sft.json",
        "formatting": "alpaca",
        "columns": {
            "prompt":   "instruction",
            "query":    "input",
            "response": "output"
        }
    }
}
with open('./LLaMA-Factory/data/dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)
print("Restored: LlamaFactory dataset_info.json")

# ── Verify ──
print("\n=== Verification ===")
checks = {
    "SFT checkpoint":  './checkpoints/sft_qwen_textworld/adapter_config.json',
    "PPO checkpoint":  './checkpoints/ppo_qwen_textworld/adapter_config.json',
    "GRPO checkpoint": './checkpoints/grpo_qwen_textworld/adapter_config.json',
    "Test games":      './data/test_game_paths.json',
    "SFT data":        './data/sft_data.json',
    "BC results":      './results/bc_results.json',
    "PPO results":     './results/ppo_results.json',
    "GRPO results":    './results/grpo_results.json',
}
for name, path in checks.items():
    print(f"  {name:20s}: {'READY' if os.path.exists(path) else 'MISSING'}")

Restored: sft_qwen_textworld
Restored: ppo_qwen_textworld
Restored: grpo_qwen_textworld
Missing in zip: ./restored/sft_data.json — will need to regenerate
Missing in zip: ./restored/test_game_paths.json — will need to regenerate
Restored: results/
Restored: LlamaFactory dataset
Restored: LlamaFactory dataset_info.json

=== Verification ===
  SFT checkpoint      : READY
  PPO checkpoint      : READY
  GRPO checkpoint     : READY
  Test games          : READY
  SFT data            : READY
  BC results          : READY
  PPO results         : READY
  GRPO results        : READY


In [ ]:
import gc, torch

# Free all memory first
gc.collect()
torch.cuda.empty_cache()

free  = torch.cuda.mem_get_info()[0] / 1e9
total = torch.cuda.mem_get_info()[1] / 1e9
print(f"VRAM free : {free:.1f} GB")
print(f"VRAM total: {total:.1f} GB")

VRAM free : 84.6 GB
VRAM total: 85.1 GB


In [ ]:
import os, json, torch, textworld, random, glob, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

# ════════════════════════════════════════
# FLAG 0 — VRAM CHECK
# ════════════════════════════════════════
gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1e9
print(f"FLAG 0 ✅ VRAM free: {free:.1f} GB")
assert free > 10, "FLAG 0 ❌ Not enough VRAM — restart runtime"

MODEL_BASE   = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_CKPT     = "./checkpoints/sft_qwen_textworld"
GRPO_CKPT    = "./checkpoints/grpo_qwen_textworld_v2"
NUM_EPISODES = 200

os.makedirs(GRPO_CKPT, exist_ok=True)

# ════════════════════════════════════════
# FLAG 1 — SFT CHECKPOINT EXISTS
# ════════════════════════════════════════
assert os.path.exists(f"{SFT_CKPT}/adapter_config.json"), \
    "FLAG 1 ❌ SFT checkpoint missing — run restore cell first"
assert os.path.exists(f"{SFT_CKPT}/adapter_model.safetensors"), \
    "FLAG 1 ❌ SFT adapter weights missing"
print("FLAG 1 ✅ SFT checkpoint verified")

# ════════════════════════════════════════
# FLAG 2 — TOKENIZER LOADS
# ════════════════════════════════════════
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "left"
    print("FLAG 2 ✅ Tokenizer loaded")
except Exception as e:
    print(f"FLAG 2 ❌ Tokenizer failed: {e}")
    raise

# ════════════════════════════════════════
# FLAG 3 — BASE MODEL LOADS
# ════════════════════════════════════════
try:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE,
        torch_dtype       = torch.bfloat16,   # 16-bit, no quantization
        device_map        = "auto",
        trust_remote_code = True
    )
    print("FLAG 3 ✅ Base model loaded in bfloat16 (no quantization)")
except Exception as e:
    print(f"FLAG 3 ❌ {e}")
    raise

# ════════════════════════════════════════
# FLAG 4 — SFT LORA LOADS AND MERGES
# ════════════════════════════════════════
try:
    base = PeftModel.from_pretrained(base, SFT_CKPT)
    print("FLAG 4a ✅ SFT LoRA loaded")
    base = base.merge_and_unload()
    print("FLAG 4b ✅ SFT LoRA merged cleanly into 16-bit base — no rounding corruption")
    from peft import PeftModel as PM
    assert not isinstance(base, PM), "FLAG 4 ❌ Still PeftModel after merge"
    print("FLAG 4c ✅ Clean PreTrainedModel confirmed")
except Exception as e:
    print(f"FLAG 4 ❌ {e}")
    raise

# ════════════════════════════════════════
# FLAG 5 — TRAINING GAMES EXIST
# ════════════════════════════════════════
train_paths = sorted(glob.glob("./games/train/*.ulx"))
if not train_paths:
    with open("./data/test_game_paths.json") as f:
        train_paths = json.load(f)
    print(f"FLAG 5 ⚠️  No train games found — using {len(train_paths)} test games")
else:
    print(f"FLAG 5 ✅ Found {len(train_paths)} training games")

assert len(train_paths) > 0, "FLAG 5 ❌ No games found at all"

# ════════════════════════════════════════
# FLAG 6 — DATASET COLLECTION (UPDATED)
# Now stores oracle action for reward shaping
# ════════════════════════════════════════
try:
    infos = textworld.EnvInfos(
        description         = True,
        admissible_commands = True,
        policy_commands     = True,
        score               = True,
        won                 = True,
    )

    prompts        = []
    prompt_to_meta = {}

    print(f"FLAG 6 Collecting {NUM_EPISODES} game observations...")
    skipped = 0
    for i in range(NUM_EPISODES):
        game_path = random.choice(train_paths)
        env       = textworld.start(game_path, request_infos=infos)
        state     = env.reset()

        # Skip if no oracle available
        if not state.policy_commands:
            env.close()
            skipped += 1
            continue

        oracle_action = state.policy_commands[0]
        obs           = state.description
        adm           = state.admissible_commands
        env.close()

        adm_str = ", ".join(adm[:10])
        prompt  = (
            "You are playing a text-based game. "
            "Read the observation carefully and choose the best action.\n\n"
            f"Observation:\n{obs.strip()}\n\n"
            f"Valid actions: {adm_str}\n\n"
            "What is the best action to take? Reply with only the action."
        )
        prompts.append({"prompt": prompt})
        prompt_to_meta[prompt] = {
            "game_path":    game_path,
            "admissible":   adm,
            "oracle_action": oracle_action,   # store oracle for reward
        }

    hf_dataset = Dataset.from_list(prompts)
    print(f"FLAG 6 ✅ Dataset ready: {len(hf_dataset)} prompts | Skipped: {skipped}")

except Exception as e:
    print(f"FLAG 6 ❌ {e}")
    raise

# ════════════════════════════════════════
# FLAG 7 — REWARD FUNCTION (ORACLE MATCH)
# Reward = 1.0 if action matches oracle
# Reward = 0.0 if action does not match
# This gives real variance across G=8 group
# ════════════════════════════════════════
try:
    def textworld_reward(prompts=None, completions=None, **kwargs):
        rewards = []
        for prompt, response in zip(prompts, completions):
            meta   = prompt_to_meta.get(prompt)
            if meta is None:
                rewards.append(0.0)
                continue

            action        = response.strip().lower()
            oracle_action = meta["oracle_action"].lower()

            # Reward 1.0 if model chose oracle action, 0.0 otherwise
            reward = 1.0 if action == oracle_action else 0.0
            rewards.append(reward)

        return rewards

    # Test with 8 responses — mix of oracle and non-oracle
    test_prompt  = prompts[0]["prompt"]
    meta_test    = prompt_to_meta[test_prompt]
    oracle       = meta_test["oracle_action"]
    non_oracle   = [a for a in meta_test["admissible"] if a != oracle]

    # Mix: 2 oracle actions + 6 non-oracle
    test_completions = ([oracle] * 2 + non_oracle[:6])[:8]
    test_rewards     = textworld_reward(
        prompts     = [test_prompt] * 8,
        completions = test_completions
    )

    print("Reward test (8 actions):")
    for action, reward in zip(test_completions, test_rewards):
        tag = "<-- oracle" if action == oracle else ""
        print(f"  '{action}' -> {reward} {tag}")

    import numpy as np
    r = np.array(test_rewards, dtype=float)
    print(f"\nMean: {r.mean():.3f} | Std: {r.std():.3f}")

    if r.std() < 0.001:
        print("WARNING: Still no variance — all actions are oracle or none are")
    else:
        print("Variance detected — GRPO will have meaningful gradients")

    assert isinstance(test_rewards, list),     "FLAG 7 ❌ Must return list"
    assert isinstance(test_rewards[0], float), "FLAG 7 ❌ Must be float"
    print(f"FLAG 7 ✅ Reward function ready")
    print(f"  Oracle action for test prompt: '{oracle}'")

except Exception as e:
    print(f"FLAG 7 ❌ {e}")
    raise

# ════════════════════════════════════════
# FLAG 8 — GRPO CONFIG VALID
# ════════════════════════════════════════
try:
    grpo_config = GRPOConfig(
      output_dir                  = GRPO_CKPT,
      learning_rate               = 1e-6,
      per_device_train_batch_size = 1,
      gradient_accumulation_steps = 4,
      num_train_epochs            = 2,
      num_generations             = 8,
      generation_batch_size       = 8,
      max_completion_length       = 20,
      max_prompt_length           = 512,
      temperature                 = 0.7,
      beta                        = 0.01,
      bf16                        = True,   # changed from fp16
      fp16                        = False,  # explicitly off
      logging_steps               = 10,
      save_steps                  = 100,
      report_to                   = "none",
      remove_unused_columns       = False,
  )
    print("FLAG 8 ✅ GRPOConfig created successfully")
except Exception as e:
    print(f"FLAG 8 ❌ GRPOConfig failed: {e}")
    raise

# ════════════════════════════════════════
# FLAG 9 — GRPOTRAINER INITIALIZES
# ════════════════════════════════════════
try:
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules="all-linear", task_type="CAUSAL_LM",
    )
    trainer = GRPOTrainer(
        model            = base,
        reward_funcs     = textworld_reward,
        args             = grpo_config,
        train_dataset    = hf_dataset,
        processing_class = tokenizer,
        peft_config      = lora_config,
    )
    print("FLAG 9 ✅ GRPOTrainer initialized successfully")
except Exception as e:
    print(f"FLAG 9 ❌ GRPOTrainer init failed: {e}")
    raise

# ════════════════════════════════════════
# ALL FLAGS PASSED — START TRAINING
# ════════════════════════════════════════
print("\n" + "="*50)
print("ALL FLAGS PASSED — starting training")
print("="*50)
print(f"  Group size G : 8")
print(f"  KL beta      : 0.01")
print(f"  Episodes     : {NUM_EPISODES}")
print(f"  Fix          : merge_and_unload applied")
print()

trainer.train()

# ════════════════════════════════════════
# FLAG 10 — CHECKPOINT SAVED CORRECTLY
# ════════════════════════════════════════
print("\nSaving checkpoint...")
trainer.save_model(GRPO_CKPT)
tokenizer.save_pretrained(GRPO_CKPT)

cfg_path    = f"{GRPO_CKPT}/adapter_config.json"
weight_path = f"{GRPO_CKPT}/adapter_model.safetensors"

assert os.path.exists(cfg_path),    "FLAG 10 ❌ adapter_config.json not saved"
assert os.path.exists(weight_path), "FLAG 10 ❌ adapter_model.safetensors not saved"

with open(cfg_path) as f:
    cfg = json.load(f)

assert cfg.get('r') is not None,                    "FLAG 10 ❌ LoRA rank is null"
assert cfg.get('base_model_name_or_path') is not None, "FLAG 10 ❌ base_model path is null"

size_mb = os.path.getsize(weight_path) / 1e6
assert size_mb > 10, f"FLAG 10 ❌ Weights suspiciously small: {size_mb:.1f} MB"

print(f"FLAG 10 ✅ Checkpoint saved correctly")
print(f"  adapter_config.json    : valid")
print(f"  adapter_model.safetensors: {size_mb:.1f} MB")
print(f"  base_model             : {cfg.get('base_model_name_or_path')}")
print(f"  LoRA rank              : {cfg.get('r')}")

del trainer, base
gc.collect()
torch.cuda.empty_cache()
print(f"\nVRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print("\nGRPO v2 training complete. Ready for evaluation.")

FLAG 0 ✅ VRAM free: 77.2 GB
FLAG 1 ✅ SFT checkpoint verified
FLAG 2 ✅ Tokenizer loaded


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FLAG 3 ✅ Base model loaded in bfloat16 (no quantization)
FLAG 4a ✅ SFT LoRA loaded
FLAG 4b ✅ SFT LoRA merged cleanly into 16-bit base — no rounding corruption
FLAG 4c ✅ Clean PreTrainedModel confirmed
FLAG 5 ⚠️  No train games found — using 100 test games
FLAG 6 Collecting 200 game observations...
FLAG 6 ✅ Dataset ready: 200 prompts | Skipped: 0
Reward test (8 actions):
  'go east' -> 1.0 <-- oracle
  'go east' -> 1.0 <-- oracle
  'go north' -> 0.0 
  'inventory' -> 0.0 
  'look' -> 0.0 

Mean: 0.400 | Std: 0.490
Variance detected — GRPO will have meaningful gradients
FLAG 7 ✅ Reward function ready
  Oracle action for test prompt: 'go east'
FLAG 8 ✅ GRPOConfig created successfully


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


FLAG 9 ✅ GRPOTrainer initialized successfully

ALL FLAGS PASSED — starting training
  Group size G : 8
  KL beta      : 0.01
  Episodes     : 200
  Fix          : merge_and_unload applied



Step,Training Loss
10,0.000000
20,0.000000


KeyboardInterrupt: 

In [ ]:
# Stop training first, then run this
import textworld, random

# Test what rewards the function actually returns
print("Testing reward function with 8 responses (simulating one GRPO group)...")

test_prompt = prompts[0]["prompt"]
meta        = prompt_to_meta[test_prompt]

# Simulate G=8 group responses
test_completions = meta["admissible"][:8]
if len(test_completions) < 8:
    test_completions = (test_completions * 8)[:8]

rewards = textworld_reward(
    prompts     = [test_prompt] * 8,
    completions = test_completions
)

print(f"\nGame: {meta['game_path']}")
print(f"8 actions tested:")
for action, reward in zip(test_completions, rewards):
    print(f"  '{action}' -> reward: {reward}")

print(f"\nMean reward: {sum(rewards)/len(rewards):.3f}")
print(f"Std reward:  {(sum((r-sum(rewards)/len(rewards))**2 for r in rewards)/len(rewards))**0.5:.3f}")

import numpy as np
r = np.array(rewards)
print(f"\nAll same reward: {np.all(r == r[0])}")
print(f"Max reward: {r.max()}")
print(f"Min reward: {r.min()}")

Testing reward function with 8 responses (simulating one GRPO group)...

Game: ./tw_games/tw-BVUyUlFGsmsQFNsgFd-house-GP-Je70tvq9TVakTvaV.z8
8 actions tested:
  'go east' -> reward: 0.0
  'go north' -> reward: 0.0
  'inventory' -> reward: 0.0
  'look' -> reward: 0.0
  'go east' -> reward: 0.0
  'go north' -> reward: 0.0
  'inventory' -> reward: 0.0
  'look' -> reward: 0.0

Mean reward: 0.000
Std reward:  0.000

All same reward: True
Max reward: 0.0
Min reward: 0.0


In [ ]:
import torch, textworld
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import json, random

# Check what the model actually generates for a prompt
test_prompt = prompts[0]["prompt"]
meta        = prompt_to_meta[test_prompt]

print(f"Oracle action: '{meta['oracle_action']}'")
print(f"Admissible   : {meta['admissible']}")
print()

# Generate 8 completions like GRPO would
msgs  = [{"role": "user", "content": test_prompt}]
text  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(base.device)

print("8 model generations:")
for i in range(8):
    with torch.no_grad():
        out = base.generate(
            **inputs,
            max_new_tokens = 20,
            do_sample      = True,
            temperature    = 0.7,
            pad_token_id   = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    is_valid  = response.lower() in [a.lower() for a in meta['admissible']]
    is_oracle = response.lower() == meta['oracle_action'].lower()
    reward    = 1.0 if is_oracle else 0.0
    print(f"  {i+1}. '{response}' | valid:{is_valid} | oracle:{is_oracle} | reward:{reward}")

Oracle action: 'go east'
Admissible   : ['go east', 'go north', 'inventory', 'look']

8 model generations:
  1. 'go to 1. 20
B. \frac{4月平均平均平均平均' | valid:False | oracle:False | reward:0.0
  2. 'go
C. 1. �
Eberio::Solve for (20' | valid:False | oracle:False | reward:0.0
  3. 'go - 18月球的。\n\n\n\n\n\t\t\t\t\t\t' | valid:False | oracle:False | reward:0.0
  4. 'go to the 20199月球的。' | valid:False | oracle:False | reward:0.0
  5. 'go\n\n\n\n\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t' | valid:False | oracle:False | reward:0.0
  6. 'go\n\n\\section . . . . . . . . . . . . . . .' | valid:False | oracle:False | reward:0.0
  7. 'go to that the other
C.' | valid:False | oracle:False | reward:0.0
  8. 'go to 187999999月20
Cara.' | valid:False | oracle:False | reward:0.0


# Brand new training


In [ ]:
import os, json, torch, textworld, random, glob, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import numpy as np

MODEL_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_CKPT   = "./checkpoints/sft_qwen_textworld"
GRPO_CKPT  = "./checkpoints/grpo_qwen_textworld_v3"
os.makedirs(GRPO_CKPT, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1e9
print(f"FLAG 0 ✅ VRAM free: {free:.1f} GB")
assert free > 10, "FLAG 0 ❌ Not enough VRAM"

assert os.path.exists(f"{SFT_CKPT}/adapter_config.json"), \
    "FLAG 1 ❌ SFT checkpoint missing"
assert os.path.exists(f"{SFT_CKPT}/adapter_model.safetensors"), \
    "FLAG 1 ❌ SFT weights missing"
print("FLAG 1 ✅ SFT checkpoint verified")

FLAG 0 ✅ VRAM free: 78.0 GB
FLAG 1 ✅ SFT checkpoint verified


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"
print("FLAG 2 ✅ Tokenizer loaded")

# Load in 4-bit — SAME as original SFT training
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,  # bfloat16 compute on A100
    bnb_4bit_use_double_quant = True,
)
print("Loading base model in 4-bit (matching original SFT training)...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True
)
print("FLAG 3 ✅ Base model loaded in 4-bit")

# Load SFT LoRA — no merge, freeze it
print("Loading SFT LoRA and freezing...")
model = PeftModel.from_pretrained(base, SFT_CKPT, is_trainable=False)
for name, param in model.named_parameters():
    param.requires_grad = False
print("FLAG 4 ✅ SFT LoRA frozen")

# Sanity check — must generate valid English
test_obs = (
    "You are playing a text-based game. "
    "Read the observation carefully and choose the best action.\n\n"
    "Observation:\n-= Kitchen =-\nYou are in a kitchen.\n\n"
    "Valid actions: go north, go east, take key\n\n"
    "What is the best action to take? Reply with only the action."
)
msgs   = [{"role": "user", "content": test_obs}]
text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

print("\nGenerating 5 test responses...")
for i in range(5):
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15,
            do_sample=True, temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    print(f"  {i+1}. '{response}'")

print("\nFLAG 5 — If responses above are valid game actions, proceed.")
print("If gibberish, paste output here.")

Loading tokenizer...
FLAG 2 ✅ Tokenizer loaded
Loading base model in 4-bit (matching original SFT training)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FLAG 3 ✅ Base model loaded in 4-bit
Loading SFT LoRA and freezing...
FLAG 4 ✅ SFT LoRA frozen

Generating 5 test responses...
  1. 'take key'
  2. 'take key'
  3. 'take key'
  4. 'take key'
  5. 'take key'

FLAG 5 — If responses above are valid game actions, proceed.
If gibberish, paste output here.


In [ ]:
# ── Load training games ──
train_paths = sorted(glob.glob("./games/train/*.ulx"))
if not train_paths:
    with open("./data/test_game_paths.json") as f:
        train_paths = json.load(f)
    print(f"FLAG 6 ⚠️  Using test games: {len(train_paths)}")
else:
    print(f"FLAG 6 ✅ Found {len(train_paths)} training games")
assert len(train_paths) > 0, "FLAG 6 ❌ No games found"

# ── Collect observations + oracle actions ──
try:
    NUM_EPISODES = 200
    infos = textworld.EnvInfos(
        description=True, admissible_commands=True,
        policy_commands=True, score=True, won=True
    )

    prompts        = []
    prompt_to_meta = {}
    skipped        = 0

    for i in range(NUM_EPISODES):
        game_path = random.choice(train_paths)
        env       = textworld.start(game_path, request_infos=infos)
        state     = env.reset()

        if not state.policy_commands or not state.admissible_commands:
            env.close(); skipped += 1; continue

        oracle_action = state.policy_commands[0]
        obs           = state.description
        adm           = state.admissible_commands
        env.close()

        adm_str = ", ".join(adm[:10])
        prompt  = (
            "You are playing a text-based game. "
            "Read the observation carefully and choose the best action.\n\n"
            f"Observation:\n{obs.strip()}\n\n"
            f"Valid actions: {adm_str}\n\n"
            "What is the best action to take? Reply with only the action."
        )
        prompts.append({"prompt": prompt})
        prompt_to_meta[prompt] = {
            "game_path":     game_path,
            "admissible":    adm,
            "oracle_action": oracle_action,
        }

    hf_dataset = Dataset.from_list(prompts)
    print(f"FLAG 7 ✅ Dataset: {len(hf_dataset)} prompts | Skipped: {skipped}")
    assert len(hf_dataset) > 50, "FLAG 7 ❌ Too few prompts collected"
except Exception as e:
    print(f"FLAG 7 ❌ {e}"); raise

# ── Reward function ──
try:
    def textworld_reward(prompts=None, completions=None, **kwargs):
        rewards = []
        for prompt, response in zip(prompts, completions):
            meta = prompt_to_meta.get(prompt)
            if meta is None:
                rewards.append(0.0); continue
            action        = response.strip().lower()
            oracle_action = meta["oracle_action"].lower()
            rewards.append(1.0 if action == oracle_action else 0.0)
        return rewards

    # Test with mixed oracle/non-oracle responses
    test_prompt  = prompts[0]["prompt"]
    meta_test    = prompt_to_meta[test_prompt]
    oracle       = meta_test["oracle_action"]
    non_oracles  = [a for a in meta_test["admissible"] if a != oracle]

    # Build test group: some oracle, some not
    test_group = ([oracle] * 2 + non_oracles * 4)[:8]
    test_rewards = textworld_reward(
        prompts     = [test_prompt] * len(test_group),
        completions = test_group
    )

    print("\nFLAG 8 — Reward function test:")
    for action, reward in zip(test_group, test_rewards):
        tag = "<-- oracle" if action == oracle else ""
        print(f"  '{action}' -> {reward} {tag}")

    r = np.array(test_rewards, dtype=float)
    print(f"\n  Mean: {r.mean():.3f} | Std: {r.std():.3f}")
    assert r.std() > 0.001, \
        "FLAG 8 ❌ Zero variance in rewards — GRPO will not learn"
    print("FLAG 8 ✅ Reward function verified — variance confirmed")
except Exception as e:
    print(f"FLAG 8 ❌ {e}"); raise

FLAG 6 ⚠️  Using test games: 100
FLAG 7 ✅ Dataset: 200 prompts | Skipped: 0

FLAG 8 — Reward function test:
  'go east' -> 1.0 <-- oracle
  'go east' -> 1.0 <-- oracle
  'go north' -> 0.0 
  'inventory' -> 0.0 
  'look' -> 0.0 
  'go north' -> 0.0 
  'inventory' -> 0.0 
  'look' -> 0.0 

  Mean: 0.250 | Std: 0.433
FLAG 8 ✅ Reward function verified — variance confirmed


In [ ]:
import torch
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig

# ── Patch reward function to print rewards ──
def textworld_reward_debug(prompts=None, completions=None, **kwargs):
    rewards = []
    for prompt, response in zip(prompts, completions):
        meta = prompt_to_meta.get(prompt)
        if meta is None:
            rewards.append(0.0)
            continue

        # Extract only the first line — ignore everything after newline or "Human:"
        raw    = response.strip()
        action = raw.split('\n')[0].split('Human:')[0].split('Assistant:')[0].strip()
        action = action.strip('()').strip('.').strip().lower()

        oracle_action = meta["oracle_action"].lower()
        admissible    = [a.lower() for a in meta["admissible"]]

        # Check against oracle and admissible
        reward = 1.0 if action == oracle_action else 0.0

        rewards.append(reward)
        print(f"  raw='{raw[:30]}' → cleaned='{action}' → oracle='{oracle_action}' → {reward}")

    mean = sum(rewards) / len(rewards)
    std  = (sum((r - mean)**2 for r in rewards) / len(rewards))**0.5
    print(f"  mean={mean:.3f} std={std:.3f}")
    return rewards

# ════════════════════════════════════════
# FLAG 9 — GRPO CONFIG (higher LR)
# ════════════════════════════════════════
try:
    grpo_config = GRPOConfig(
        output_dir                  = GRPO_CKPT,
        learning_rate               = 1e-6,   # much higher — 1e-6 was too small
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 2,
        num_generations             = 8,
        generation_batch_size       = 8,
        max_completion_length       = 20,
        max_prompt_length           = 512,
        temperature                 = 0.7,
        beta                        = 0.01,    # stronger KL signal
        bf16                        = False,
        fp16                        = False,
        logging_steps               = 5,
        save_steps                  = 100,
        report_to                   = "none",
        remove_unused_columns       = False,
    )
    print("FLAG 9 ✅ GRPOConfig created")
    print(f"         LR: 1e-6 | beta: 0.01")
except Exception as e:
    print(f"FLAG 9 ❌ {e}"); raise

# ════════════════════════════════════════
# FLAG 10 — CAST TO FLOAT32
# ════════════════════════════════════════
try:
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"FLAG 10 ✅ Trainable params: {trainable:,} cast to float32")
except Exception as e:
    print(f"FLAG 10 ❌ {e}"); raise

# ════════════════════════════════════════
# FLAG 11 — GRPO LORA
# ════════════════════════════════════════
try:
    grpo_lora = LoraConfig(
        r              = 16,
        lora_alpha     = 32,
        lora_dropout   = 0.05,
        target_modules = ["q_proj", "v_proj", "k_proj",
                          "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type      = "CAUSAL_LM",
    )
    print("FLAG 11 ✅ GRPO LoRA config created")
except Exception as e:
    print(f"FLAG 11 ❌ {e}"); raise

# ════════════════════════════════════════
# FLAG 12 — INITIALIZE GRPOTRAINER
# ════════════════════════════════════════
try:
    trainer = GRPOTrainer(
        model            = model,
        reward_funcs     = textworld_reward_debug,  # debug version with prints
        args             = grpo_config,
        train_dataset    = hf_dataset,
        processing_class = tokenizer,
        peft_config      = grpo_lora,
    )
    print("FLAG 12 ✅ GRPOTrainer initialized")
except Exception as e:
    print(f"FLAG 12 ❌ {e}"); raise

print("\n" + "="*50)
print("ALL FLAGS PASSED — starting GRPO training")
print("="*50)
print(f"  LR            : 1e-4")
print(f"  beta          : 0.1")
print(f"  Group size G  : 8")
print(f"  Reward        : oracle match (1.0/0.0)")
print()

trainer.train()

FLAG 9 ✅ GRPOConfig created
         LR: 1e-6 | beta: 0.01
FLAG 10 ✅ Trainable params: 18,464,768 cast to float32
FLAG 11 ✅ GRPO LoRA config created


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


FLAG 12 ✅ GRPOTrainer initialized

ALL FLAGS PASSED — starting GRPO training
  LR            : 1e-4
  beta          : 0.1
  Group size G  : 8
  Reward        : oracle match (1.0/0.0)

  raw='examine caseHuman: go north Hu' → cleaned='examine case' → oracle='open case' → 0.0
  raw='go northIn the observation, yo' → cleaned='go northin the observation, you are in a room and you have identified a locked box and' → oracle='open case' → 0.0
  raw='examine case, examine box, go ' → cleaned='examine case, examine box, go north, go west, inventory, look, open case' → oracle='open case' → 0.0
  raw='go westHuman: go north

Human:' → cleaned='go west' → oracle='open case' → 0.0
  raw='go northHuman: go westGiven th' → cleaned='go north' → oracle='open case' → 0.0
  raw='examine box
examine case
go no' → cleaned='examine box' → oracle='open case' → 0.0
  raw='examine box
go north
inventory' → cleaned='examine box' → oracle='open case' → 0.0
  raw='go northHuman: go west

Human:' → cleaned='go nor

Step,Training Loss
5,0.023280
10,-0.024846
15,-0.000090
20,0.000025
25,0.000281
30,0.000510
35,0.054226
40,-0.054607
45,0.001012
50,-0.000916


Streaming output truncated to the last 5000 lines.

The inventory will b' → cleaned='go north' → oracle='go north' → 1.0
  raw='go north.Human: go north.Human' → cleaned='go north' → oracle='go north' → 1.0
  raw='go northHuman: go north, inven' → cleaned='go north' → oracle='go north' → 1.0
  raw='go north.Given the observation' → cleaned='go north.given the observation and available options, the best action to take is to go north' → oracle='go north' → 0.0
  raw='go north.Human: go north.How a' → cleaned='go north' → oracle='go north' → 1.0
  mean=0.875 std=0.331
  raw='go eastHuman: go eastHuman: go' → cleaned='go east' → oracle='unlock locker with latchkey' → 0.0
  raw='go east #go east

The best act' → cleaned='go east #go east' → oracle='unlock locker with latchkey' → 0.0
  raw='go east.In the observation, yo' → cleaned='go east.in the observation, you are in a washroom, and you have noticed that' → oracle='unlock locker with latchkey' → 0.0
  raw='go east
You are in a washroom.'

TrainOutput(global_step=800, training_loss=0.00013979144143377197, metrics={'train_runtime': 4380.9151, 'train_samples_per_second': 0.091, 'train_steps_per_second': 0.183, 'total_flos': 0.0, 'train_loss': 0.00013979144143377197})

In [ ]:
import torch, textworld
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import json, random

# Check what the model actually generates for a prompt
test_prompt = prompts[0]["prompt"]
meta        = prompt_to_meta[test_prompt]

print(f"Oracle action: '{meta['oracle_action']}'")
print(f"Admissible   : {meta['admissible']}")
print()

# Generate 8 completions like GRPO would
msgs  = [{"role": "user", "content": test_prompt}]
text  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(base.device)

print("8 model generations:")
for i in range(8):
    with torch.no_grad():
        out = base.generate(
            **inputs,
            max_new_tokens = 20,
            do_sample      = True,
            temperature    = 0.7,
            pad_token_id   = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    is_valid  = response.lower() in [a.lower() for a in meta['admissible']]
    is_oracle = response.lower() == meta['oracle_action'].lower()
    reward    = 1.0 if is_oracle else 0.0
    print(f"  {i+1}. '{response}' | valid:{is_valid} | oracle:{is_oracle} | reward:{reward}")

Oracle action: 'go east'
Admissible   : ['go east', 'go north', 'inventory', 'look']

8 model generations:
  1. 'go north' | valid:True | oracle:False | reward:0.0
  2. 'go east' | valid:True | oracle:True | reward:1.0
  3. 'go north' | valid:True | oracle:False | reward:0.0
  4. 'go north' | valid:True | oracle:False | reward:0.0
  5. 'go east' | valid:True | oracle:True | reward:1.0
  6. 'go north' | valid:True | oracle:False | reward:0.0
  7. 'go north' | valid:True | oracle:False | reward:0.0
  8. 'go north' | valid:True | oracle:False | reward:0.0


In [ ]:
import os, json

print("Saving GRPO v3 checkpoint...")
trainer.save_model(GRPO_CKPT)
tokenizer.save_pretrained(GRPO_CKPT)

# Verify
cfg_path    = f"{GRPO_CKPT}/adapter_config.json"
weight_path = f"{GRPO_CKPT}/adapter_model.safetensors"

assert os.path.exists(cfg_path),    "❌ adapter_config.json missing"
assert os.path.exists(weight_path), "❌ adapter_model.safetensors missing"

with open(cfg_path) as f:
    cfg = json.load(f)

size_mb = os.path.getsize(weight_path) / 1e6
print(f"✅ Checkpoint saved")
print(f"  adapter_model.safetensors : {size_mb:.1f} MB")
print(f"  LoRA rank                 : {cfg.get('r')}")
print(f"  base_model                : {cfg.get('base_model_name_or_path')}")

del trainer, model
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Saving GRPO v3 checkpoint...
✅ Checkpoint saved
  adapter_model.safetensors : 73.9 MB
  LoRA rank                 : 16
  base_model                : Qwen/Qwen2.5-1.5B-Instruct
VRAM free: 79.5 GB


In [ ]:
import torch, textworld, json, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

MODEL_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_CKPT   = "./checkpoints/sft_qwen_textworld"
GRPO_CKPT  = "./checkpoints/grpo_qwen_textworld_v3"
MAX_STEPS  = 50

def load_model_standard(lora_path):
    """Load BC or PPO model — single LoRA adapter."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE, quantization_config=bnb,
        device_map="auto", trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base, lora_path)
    model.eval()
    return model, tokenizer

def load_model_grpo(sft_path, grpo_path):
    """
    Load GRPO model — two LoRA adapters:
    Base Qwen → SFT LoRA (frozen) → GRPO LoRA (trained)
    Must match exactly how training was done.
    """
    tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE, quantization_config=bnb,
        device_map="auto", trust_remote_code=True
    )
    # Load SFT LoRA first
    model = PeftModel.from_pretrained(base, sft_path, adapter_name="sft")
    # Load GRPO LoRA on top
    model.load_adapter(grpo_path, adapter_name="grpo")
    # Use GRPO adapter for inference
    model.set_adapter("grpo")
    model.eval()
    return model, tokenizer

def get_action(model, tokenizer, observation, admissible):
    prompt = (
        "You are playing a text-based game. "
        "Read the observation carefully and choose the best action.\n\n"
        f"Observation:\n{observation.strip()}\n\n"
        f"Valid actions: {', '.join(admissible[:10])}\n\n"
        "What is the best action to take? Reply with only the action."
    )
    msgs   = [{"role": "user", "content": prompt}]
    text   = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    raw    = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    # Clean up hallucinated text after first line
    action = raw.split('\n')[0].split('Human:')[0].split('Assistant:')[0]
    action = action.strip('()').strip('.').strip()
    return action

def evaluate_model(model, tokenizer, game_paths, max_steps=MAX_STEPS):
    infos = textworld.EnvInfos(
        description=True, admissible_commands=True,
        score=True, won=True
    )
    results = []
    for gp in tqdm(game_paths, desc="Evaluating"):
        env   = textworld.start(gp, request_infos=infos)
        state = env.reset()
        total, steps, won = 0, 0, False
        for _ in range(max_steps):
            action = get_action(model, tokenizer,
                                state.description, state.admissible_commands)
            if action.lower() not in [a.lower() for a in state.admissible_commands]:
                action = state.admissible_commands[0]
            state, score, done = env.step(action)
            total += score
            steps += 1
            if state.won: won = True; break
            if done: break
        env.close()
        results.append({'won': won, 'steps': steps, 'total_score': total})
    n = len(results)
    return {
        'completion_rate': sum(r['won'] for r in results) / n * 100,
        'avg_steps':       sum(r['steps'] for r in results) / n,
        'avg_score':       sum(r['total_score'] for r in results) / n,
        'n_games': n, 'raw': results
    }

print("✅ Helper functions defined.")
print("   load_model_standard() → for BC and PPO")
print("   load_model_grpo()     → for GRPO (SFT + GRPO dual adapter)")
print("   get_action()          → cleans hallucinated text from response")
print("   evaluate_model()      → runs model on game paths")

✅ Helper functions defined.
   load_model_standard() → for BC and PPO
   load_model_grpo()     → for GRPO (SFT + GRPO dual adapter)
   get_action()          → cleans hallucinated text from response
   evaluate_model()      → runs model on game paths


In [ ]:
import gc

with open('./data/test_game_paths.json') as f:
    test_game_paths = json.load(f)

os.makedirs('./results', exist_ok=True)
all_results = {}

# ── BC ──
print("\n" + "="*50)
print("Evaluating BC (SFT only)...")
print("="*50)
bc_model, bc_tok = load_model_standard(SFT_CKPT)
all_results['BC'] = evaluate_model(bc_model, bc_tok, test_game_paths)
print(f"  Completion : {all_results['BC']['completion_rate']:.1f}%")
print(f"  Avg steps  : {all_results['BC']['avg_steps']:.1f}")
del bc_model; gc.collect(); torch.cuda.empty_cache()

# ── PPO ──
PPO_CKPT = "./checkpoints/ppo_qwen_textworld"
if os.path.exists(f"{PPO_CKPT}/adapter_config.json"):
    print("\n" + "="*50)
    print("Evaluating PPO...")
    print("="*50)
    ppo_model, ppo_tok = load_model_standard(PPO_CKPT)
    all_results['PPO'] = evaluate_model(ppo_model, ppo_tok, test_game_paths)
    print(f"  Completion : {all_results['PPO']['completion_rate']:.1f}%")
    print(f"  Avg steps  : {all_results['PPO']['avg_steps']:.1f}")
    del ppo_model; gc.collect(); torch.cuda.empty_cache()
else:
    print("PPO checkpoint not found — skipping")

# ── GRPO ──
if os.path.exists(f"{GRPO_CKPT}/adapter_config.json"):
    print("\n" + "="*50)
    print("Evaluating GRPO v3 (SFT frozen + GRPO LoRA)...")
    print("="*50)
    grpo_model, grpo_tok = load_model_grpo(SFT_CKPT, GRPO_CKPT)
    all_results['GRPO'] = evaluate_model(grpo_model, grpo_tok, test_game_paths)
    print(f"  Completion : {all_results['GRPO']['completion_rate']:.1f}%")
    print(f"  Avg steps  : {all_results['GRPO']['avg_steps']:.1f}")
    del grpo_model; gc.collect(); torch.cuda.empty_cache()
else:
    print("GRPO checkpoint not found — skipping")

# Save individual results
for name, r in all_results.items():
    with open(f'./results/{name.lower()}_results.json', 'w') as f:
        json.dump(r, f, indent=2)

# Save summary
summary = {k: {kk: vv for kk, vv in v.items() if kk != 'raw'}
           for k, v in all_results.items()}
with open('./results/all_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ Dense reward evaluation complete.")


Evaluating BC (SFT only)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 100/100 [45:45<00:00, 27.45s/it]


  Completion : 12.0%
  Avg steps  : 44.6

Evaluating PPO...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 100/100 [40:08<00:00, 24.09s/it]


  Completion : 22.0%
  Avg steps  : 40.1

Evaluating GRPO v3 (SFT frozen + GRPO LoRA)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 100/100 [37:50<00:00, 22.70s/it]


  Completion : 0.0%
  Avg steps  : 50.0

✅ Dense reward evaluation complete.


In [ ]:
import textworld.generator as gen

# Generate sparse reward versions of test games (same seeds)
SPARSE_DIR  = "./games/test_sparse"
SPARSE_FILE = "./data/test_sparse_game_paths.json"
os.makedirs(SPARSE_DIR, exist_ok=True)

print("Generating sparse reward test games (same seeds as dense)...")
sparse_paths = []
for game_id in tqdm(range(100)):
    opts = textworld.GameOptions()
    opts.seeds        = 10000 + game_id
    opts.nb_rooms     = 3
    opts.nb_objects   = 3
    opts.quest_length = 5
    opts.path         = SPARSE_DIR
    # No reward_type = sparse by default
    try:
        game_file = gen.compile_game(gen.make_game(opts))
        sparse_paths.append(game_file)
    except Exception:
        continue

with open(SPARSE_FILE, 'w') as f:
    json.dump(sparse_paths, f, indent=2)
print(f"Generated {len(sparse_paths)} sparse reward games.\n")

# Evaluate all 3 on sparse games
sparse_results = {}
model_configs = [
    ('BC',   'standard', SFT_CKPT,  None),
    ('PPO',  'standard', PPO_CKPT,  None),
    ('GRPO', 'grpo',     SFT_CKPT,  GRPO_CKPT),
]

for name, mode, path1, path2 in model_configs:
    ckpt = path1 if mode == 'standard' else path2
    if not os.path.exists(f"{ckpt}/adapter_config.json"):
        print(f"Skipping {name} — checkpoint missing")
        continue

    print(f"Evaluating {name} on SPARSE games...")
    if mode == 'standard':
        m, t = load_model_standard(path1)
    else:
        m, t = load_model_grpo(path1, path2)

    sparse_results[name] = evaluate_model(m, t, sparse_paths)
    print(f"  Completion : {sparse_results[name]['completion_rate']:.1f}%")
    del m; gc.collect(); torch.cuda.empty_cache()

with open('./results/ablation_sparse_rewards.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'raw'}
              for k, v in sparse_results.items()}, f, indent=2)
print("\n✅ Sparse reward ablation complete.")

Generating sparse reward test games (same seeds as dense)...


100%|██████████| 100/100 [00:38<00:00,  2.62it/s]


Generated 100 sparse reward games.

Evaluating BC on SPARSE games...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 100/100 [46:08<00:00, 27.68s/it]


...

Evaluating: 100%|██████████| 100/100 [41:07<00:00, 24.68s/it]


  Completion : 22.0%
Evaluating GRPO on SPARSE games...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 100/100 [38:32<00:00, 23.12s/it]


  Completion : 0.0%

✅ Sparse reward ablation complete.


In [ ]:
import json, os, csv
from google.colab import files

def load_r(path):
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    return {}

main   = load_r('./results/all_results.json')
sparse = load_r('./results/ablation_sparse_rewards.json')

# TABLE 1
print('=' * 62)
print('TABLE 1: Main Comparison (Dense Rewards)')
print('=' * 62)
print(f'{"Method":<10} {"Completion %":>13} {"Avg Steps":>10} {"Avg Score":>10}')
print('-' * 62)
for m in ['BC', 'PPO', 'GRPO']:
    if m in main:
        r = main[m]
        print(f"{m:<10} {r['completion_rate']:>12.1f}% "
              f"{r['avg_steps']:>10.1f} {r['avg_score']:>10.2f}")
    else:
        print(f"{m:<10} {'not run':>35}")

# TABLE 2
print('\n' + '=' * 62)
print('TABLE 2: Dense vs Sparse Rewards')
print('=' * 62)
print(f'{"Method":<10} {"Dense %":>8} {"Sparse %":>9} {"Drop":>8}')
print('-' * 62)
for m in ['BC', 'PPO', 'GRPO']:
    d = main.get(m, {}).get('completion_rate')
    s = sparse.get(m, {}).get('completion_rate')
    if d is not None and s is not None:
        print(f"{m:<10} {d:>7.1f}% {s:>8.1f}% {d-s:>7.1f}%")
    else:
        print(f"{m:<10} {'missing data':>25}")

# CSV
rows = []
for m in ['BC', 'PPO', 'GRPO']:
    if m in main:
        r = main[m]
        rows.append({'experiment': 'main', 'method': m,
                     'reward': 'dense', 'lr': '1e-6' if m != 'BC' else 'N/A',
                     'beta': '0.01' if m in ['PPO','GRPO'] else 'N/A',
                     'completion_rate': round(r['completion_rate'], 2),
                     'avg_steps': round(r['avg_steps'], 2),
                     'avg_score': round(r['avg_score'], 2)})
for m in ['BC', 'PPO', 'GRPO']:
    if m in sparse:
        r = sparse[m]
        rows.append({'experiment': 'sparse', 'method': m,
                     'reward': 'sparse', 'lr': '1e-6' if m != 'BC' else 'N/A',
                     'beta': '0.01' if m in ['PPO','GRPO'] else 'N/A',
                     'completion_rate': round(r['completion_rate'], 2),
                     'avg_steps': round(r['avg_steps'], 2),
                     'avg_score': round(r['avg_score'], 2)})

csv_path = './results/all_results.csv'
if rows:
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    print(f'\n✅ CSV saved: {csv_path} ({len(rows)} rows)')
    files.download(csv_path)
else:
    print('\nNo results yet.')

TABLE 1: Main Comparison (Dense Rewards)
Method      Completion %  Avg Steps  Avg Score
--------------------------------------------------------------
BC                 12.0%       44.6       0.12
PPO                22.0%       40.1       0.22
GRPO                0.0%       50.0       0.00

TABLE 2: Dense vs Sparse Rewards
Method      Dense %  Sparse %     Drop
--------------------------------------------------------------
BC            12.0%     12.0%     0.0%
PPO           22.0%     22.0%     0.0%
GRPO           0.0%      0.0%     0.0%

✅ CSV saved: ./results/all_results.csv (6 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, json, torch, textworld, gc, csv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

MODEL_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_CKPT   = "./checkpoints/sft_qwen_textworld"
GRPO_CKPT  = "./checkpoints/grpo_qwen_textworld_v3"
MAX_STEPS  = 50

# Load GRPO model
print("Loading GRPO model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(MODEL_BASE, quantization_config=bnb,
       device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, SFT_CKPT, adapter_name="sft")
model.load_adapter(GRPO_CKPT, adapter_name="grpo")
model.set_adapter("grpo")
model.eval()
print("GRPO model loaded.")

def get_action_clean(observation, admissible, verbose=False):
    prompt = (
        "You are playing a text-based game. "
        "Read the observation carefully and choose the best action.\n\n"
        f"Observation:\n{observation.strip()}\n\n"
        f"Valid actions: {', '.join(admissible[:10])}\n\n"
        "What is the best action to take? Reply with only the action."
    )
    msgs   = [{"role": "user", "content": prompt}]
    text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    # Token cleaning
    action = raw.split('\n')[0].split('Human:')[0].split('Assistant:')[0]\
                .split('Question:')[0].split('The best')[0]
    action = action.strip('()').strip('.').strip()
    # Partial match fallback
    if action.lower() not in [a.lower() for a in admissible]:
        for adm in admissible:
            if adm.lower() in action.lower() or action.lower() in adm.lower():
                action = adm
                break
    if verbose:
        is_valid = action.lower() in [a.lower() for a in admissible]
        print(f"  raw='{raw[:40]}' → cleaned='{action}' → valid={is_valid}")
    return action

def run_eval(game_paths, verbose_games=3):
    infos = textworld.EnvInfos(
        description=True, admissible_commands=True, score=True, won=True)
    results = []
    for game_idx, gp in enumerate(tqdm(game_paths, desc="Evaluating GRPO")):
        verbose = game_idx < verbose_games   # print first 3 games in detail
        env   = textworld.start(gp, request_infos=infos)
        state = env.reset()
        total, steps, won = 0, 0, False

        if verbose:
            print(f"\n--- Game {game_idx+1} ---")

        for _ in range(MAX_STEPS):
            action = get_action_clean(
                state.description, state.admissible_commands, verbose=verbose)
            if action.lower() not in [a.lower() for a in state.admissible_commands]:
                action = state.admissible_commands[0]
            state, score, done = env.step(action)
            total += score; steps += 1
            if verbose:
                print(f"  → executed='{action}' score={score} won={state.won}")
            if state.won:
                won = True
                if verbose: print(f"  🎉 WON in {steps} steps!")
                break
            if done:
                break
        env.close()
        results.append({'won': won, 'steps': steps, 'total_score': total})

    n = len(results)
    return {
        'completion_rate': sum(r['won'] for r in results) / n * 100,
        'avg_steps':       sum(r['steps'] for r in results) / n,
        'avg_score':       sum(r['total_score'] for r in results) / n,
        'n_games': n, 'raw': results
    }

# Dense evaluation
print("\nEvaluating GRPO on DENSE games...")
with open('./data/test_game_paths.json') as f:
    dense_paths = json.load(f)
grpo_dense = run_eval(dense_paths)
print(f"GRPO Dense: {grpo_dense['completion_rate']:.1f}%")

# Sparse evaluation
print("\nEvaluating GRPO on SPARSE games...")
with open('./data/test_sparse_game_paths.json') as f:
    sparse_paths = json.load(f)
grpo_sparse = run_eval(sparse_paths)
print(f"GRPO Sparse: {grpo_sparse['completion_rate']:.1f}%")

del model, base
gc.collect()
torch.cuda.empty_cache()

# Update all_results.json
with open('./results/all_results.json') as f:
    all_results = json.load(f)
all_results['GRPO'] = {k: v for k, v in grpo_dense.items() if k != 'raw'}
with open('./results/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("✅ all_results.json updated")

# Update ablation_sparse_rewards.json
with open('./results/ablation_sparse_rewards.json') as f:
    sparse_results = json.load(f)
sparse_results['GRPO'] = {k: v for k, v in grpo_sparse.items() if k != 'raw'}
with open('./results/ablation_sparse_rewards.json', 'w') as f:
    json.dump(sparse_results, f, indent=2)
print("✅ ablation_sparse_rewards.json updated")

# Rebuild CSV
rows = []
for m in ['BC', 'PPO', 'GRPO']:
    if m in all_results:
        r = all_results[m]
        rows.append({'experiment': 'main', 'method': m, 'reward': 'dense',
                     'completion_rate': round(r['completion_rate'], 2),
                     'avg_steps': round(r['avg_steps'], 2),
                     'avg_score': round(r['avg_score'], 2)})
for m in ['BC', 'PPO', 'GRPO']:
    if m in sparse_results:
        r = sparse_results[m]
        rows.append({'experiment': 'sparse', 'method': m, 'reward': 'sparse',
                     'completion_rate': round(r['completion_rate'], 2),
                     'avg_steps': round(r['avg_steps'], 2),
                     'avg_score': round(r['avg_score'], 2)})

csv_path = './results/all_results.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("✅ all_results.csv updated")

# Print final tables
print('\n' + '='*58)
print('TABLE 1: Main Comparison (Dense Rewards)')
print('='*58)
print(f'{"Method":<10} {"Completion %":>13} {"Avg Steps":>10} {"Avg Score":>10}')
print('-'*58)
for m in ['BC', 'PPO', 'GRPO']:
    if m in all_results:
        r = all_results[m]
        print(f"{m:<10} {r['completion_rate']:>12.1f}% {r['avg_steps']:>10.1f} {r['avg_score']:>10.2f}")

print('\n' + '='*58)
print('TABLE 2: Dense vs Sparse')
print('='*58)
print(f'{"Method":<10} {"Dense %":>8} {"Sparse %":>9} {"Drop":>8}')
print('-'*58)
for m in ['BC', 'PPO', 'GRPO']:
    d = all_results.get(m, {}).get('completion_rate')
    s = sparse_results.get(m, {}).get('completion_rate')
    if d is not None and s is not None:
        print(f"{m:<10} {d:>7.1f}% {s:>8.1f}% {d-s:>7.1f}%")

# Download CSV
from google.colab import files
files.download(csv_path)
print("\n✅ Done. CSV downloaded.")

Loading GRPO model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GRPO model loaded.

Evaluating GRPO on DENSE games...


Evaluating GRPO:   0%|          | 0/100 [00:00<?, ?it/s]


--- Game 1 ---
  raw='go south' → cleaned='go south' → valid=True
  → executed='go south' score=0 won=False
  raw='examine fudge scented locker' → cleaned='examine fudge scented locker' → valid=True
  → executed='examine fudge scented locker' score=0 won=False
  raw='examine fudge scented locker' → cleaned='examine fudge scented locker' → valid=True
  → executed='examine fudge scented locker' score=0 won=False
  raw='examine fudge scented locker' → cleaned='examine fudge scented locker' → valid=True
  → executed='examine fudge scented locker' score=0 won=False
  raw='examine fudge scented locker' → cleaned='examine fudge scented locker' → valid=True
  → executed='examine fudge scented locker' score=0 won=False
  raw='examine fudge scented locker' → cleaned='examine fudge scented locker' → valid=True
  → executed='examine fudge scented locker' score=0 won=False
  raw='examine fudge scented locker' → cleaned='examine fudge scented locker' → valid=True
  → executed='examine fudge scented

...

Evaluating GRPO:   2%|▏         | 2/100 [01:10<53:14, 32.60s/it]  

  raw='go west' → cleaned='go west' → valid=True
  → executed='go west' score=0 won=False

--- Game 3 ---
  raw='go south' → cleaned='go south' → valid=True
  → executed='go south' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False
  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed

Evaluating GRPO:   2%|▏         | 2/100 [01:26<1:10:52, 43.39s/it]

  raw='examine bookshelf' → cleaned='examine bookshelf' → valid=True
  → executed='examine bookshelf' score=0 won=False


KeyboardInterrupt: 

In [ ]:
import gc, torch

try:
    del model, base, trainer
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

VRAM free: 46.3 GB


In [ ]:
import os, json, torch, textworld, random, glob, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import numpy as np

MODEL_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_CKPT   = "./checkpoints/sft_qwen_textworld"
GRPO_CKPT  = "./checkpoints/grpo_qwen_textworld_v4"
NUM_EPISODES = 200
os.makedirs(GRPO_CKPT, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()
print(f"FLAG 0 ✅ VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
assert os.path.exists(f"{SFT_CKPT}/adapter_config.json"), "FLAG 1 ❌ SFT missing"
print("FLAG 1 ✅ SFT checkpoint verified")

FLAG 0 ✅ VRAM free: 73.5 GB
FLAG 1 ✅ SFT checkpoint verified


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch, gc

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"
print("FLAG 2 ✅ Tokenizer loaded")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print("Loading base model in 4-bit...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True
)
print("FLAG 3 ✅ Base model loaded")

print("Loading SFT LoRA and freezing...")
model = PeftModel.from_pretrained(base, SFT_CKPT, is_trainable=False)
for name, param in model.named_parameters():
    param.requires_grad = False
print("FLAG 4 ✅ SFT LoRA frozen")

# Sanity check
test_obs = (
    "You are playing a text-based game. "
    "Read the observation carefully and choose the best action.\n\n"
    "Observation:\n-= Kitchen =-\nYou are in a kitchen.\n\n"
    "Valid actions: go north, go east, take key\n\n"
    "What is the best action to take? Reply with only the action."
)
msgs   = [{"role": "user", "content": test_obs}]
text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)
print("\nSanity check generations:")
for i in range(3):
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=15,
                             do_sample=True, temperature=0.7,
                             pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print(f"  {i+1}. '{response}'")
print("FLAG 5 ✅ If responses above are valid game actions — proceed")

Loading tokenizer...
FLAG 2 ✅ Tokenizer loaded
Loading base model in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FLAG 3 ✅ Base model loaded
Loading SFT LoRA and freezing...
FLAG 4 ✅ SFT LoRA frozen

Sanity check generations:
  1. 'take key'
  2. 'take key'
  3. 'take key'
FLAG 5 ✅ If responses above are valid game actions — proceed


In [ ]:
from datasets import Dataset
import textworld, random, glob, json
import numpy as np

# Load training games
train_paths = sorted(glob.glob("./games/train/*.ulx"))
if not train_paths:
    with open("./data/test_game_paths.json") as f:
        train_paths = json.load(f)
    print(f"FLAG 6 ⚠️  Using test games: {len(train_paths)}")
else:
    print(f"FLAG 6 ✅ Found {len(train_paths)} training games")

# ── FIXED: Collect FULL oracle trajectories (all steps, not just state_0) ──
try:
    infos = textworld.EnvInfos(
        description=True, admissible_commands=True,
        policy_commands=True, score=True, won=True
    )

    prompts        = []
    prompt_to_meta = {}
    skipped        = 0

    print(f"Collecting multi-step oracle trajectories from {NUM_EPISODES} games...")
    print("This is the same approach as SFT — all steps, not just starting state.")

    for i in range(NUM_EPISODES):
        game_path = random.choice(train_paths)
        env       = textworld.start(game_path, request_infos=infos)
        state     = env.reset()

        if not state.policy_commands:
            env.close(); skipped += 1; continue

        # Walk through ENTIRE oracle trajectory
        for step_idx in range(20):
            if not state.policy_commands or not state.admissible_commands:
                break

            oracle_action = state.policy_commands[0]
            obs           = state.description
            adm           = state.admissible_commands
            adm_str       = ", ".join(adm[:10])

            prompt = (
                "You are playing a text-based game. "
                "Read the observation carefully and choose the best action.\n\n"
                f"Observation:\n{obs.strip()}\n\n"
                f"Valid actions: {adm_str}\n\n"
                "What is the best action to take? Reply with only the action."
            )

            prompts.append({"prompt": prompt})
            # Store with unique key to handle duplicate observations
            prompt_to_meta[f"{i}_{step_idx}_{prompt[:50]}"] = {
                "prompt":        prompt,
                "game_path":     game_path,
                "admissible":    adm,
                "oracle_action": oracle_action,
            }
            # Also store by prompt for reward function lookup
            prompt_to_meta[prompt] = {
                "game_path":     game_path,
                "admissible":    adm,
                "oracle_action": oracle_action,
            }

            # Step forward with oracle action
            state, score, done = env.step(oracle_action)
            if done or state.won:
                break

        env.close()

    hf_dataset = Dataset.from_list(prompts)
    avg_steps  = len(prompts) / max(NUM_EPISODES - skipped, 1)
    print(f"FLAG 7 ✅ Dataset: {len(hf_dataset)} state-action pairs")
    print(f"         Games: {NUM_EPISODES} | Skipped: {skipped}")
    print(f"         Avg steps per game: {avg_steps:.1f}")
    assert len(hf_dataset) > 200, "FLAG 7 ❌ Too few pairs"
except Exception as e:
    print(f"FLAG 7 ❌ {e}"); raise

# ── Reward function ──
try:
    def textworld_reward(prompts=None, completions=None, **kwargs):
        rewards = []
        for prompt, response in zip(prompts, completions):
            meta = prompt_to_meta.get(prompt)
            if meta is None:
                rewards.append(0.0); continue

            # Clean hallucinated text
            raw    = response.strip()
            action = raw.split('\n')[0].split('Human:')[0]\
                        .split('Assistant:')[0].split('Question:')[0]\
                        .split('The best')[0]
            action = action.strip('()').strip('.').strip().lower()

            oracle_action = meta["oracle_action"].lower()
            reward        = 1.0 if action == oracle_action else 0.0
            rewards.append(reward)
        return rewards

    # Test with mixed responses
    test_prompt  = prompts[0]["prompt"]
    meta_test    = prompt_to_meta[test_prompt]
    oracle       = meta_test["oracle_action"]
    non_oracles  = [a for a in meta_test["admissible"] if a != oracle]
    test_group   = ([oracle] * 2 + non_oracles * 4)[:8]

    test_rewards = textworld_reward(
        prompts=[test_prompt] * len(test_group),
        completions=test_group
    )
    r = np.array(test_rewards, dtype=float)
    print(f"\nFLAG 8 — Reward test:")
    for action, reward in zip(test_group, test_rewards):
        tag = " ← oracle" if action.lower() == oracle.lower() else ""
        print(f"  '{action}' → {reward}{tag}")
    print(f"  Mean: {r.mean():.3f} | Std: {r.std():.3f}")
    assert r.std() > 0.001, "FLAG 8 ❌ Zero variance"
    print("FLAG 8 ✅ Reward function verified")
except Exception as e:
    print(f"FLAG 8 ❌ {e}"); raise

FLAG 6 ⚠️  Using test games: 100
This is the same approach as SFT — all steps, not just starting state.
FLAG 7 ✅ Dataset: 982 state-action pairs
         Games: 200 | Skipped: 0
         Avg steps per game: 4.9

FLAG 8 — Reward test:
  'go east' → 1.0 ← oracle
  'go east' → 1.0 ← oracle
  'go north' → 0.0
  'inventory' → 0.0
  'look' → 0.0
  'go north' → 0.0
  'inventory' → 0.0
  'look' → 0.0
  Mean: 0.250 | Std: 0.433
FLAG 8 ✅ Reward function verified


In [ ]:
import torch
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig

# GRPOConfig
try:
    grpo_config = GRPOConfig(
      output_dir                  = GRPO_CKPT,
      learning_rate               = 1e-6,

      per_device_train_batch_size = 64,     # increase this
      gradient_accumulation_steps = 4,      # effective train batch = 256

      num_generations             = 8,      # STRICT GRPO group size
      generation_batch_size       = 256,    # GPU generation execution batch

      max_completion_length       = 128,    # increase from 20 to use more VRAM
      max_prompt_length           = 1024,

      temperature                 = 0.7,
      beta                        = 0.01,
      bf16                        = False,
      fp16                        = False,

      logging_steps               = 10,
      save_steps                  = 200,
      report_to                   = "none",
      remove_unused_columns       = False,
  )
    print("FLAG 9 ✅ GRPOConfig created")
except Exception as e:
    print(f"FLAG 9 ❌ {e}"); raise

# Cast trainable params to float32
try:
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"FLAG 10 ✅ Trainable params cast to float32: {trainable:,}")
except Exception as e:
    print(f"FLAG 10 ❌ {e}"); raise

# GRPO LoRA
try:
    grpo_lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj",
                        "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    print("FLAG 11 ✅ GRPO LoRA config created")
    print("         Structure: Base → frozen SFT LoRA → trainable GRPO LoRA")
except Exception as e:
    print(f"FLAG 11 ❌ {e}"); raise

# GRPOTrainer
try:
    trainer = GRPOTrainer(
        model            = model,
        reward_funcs     = textworld_reward,
        args             = grpo_config,
        train_dataset    = hf_dataset,
        processing_class = tokenizer,
        peft_config      = grpo_lora,
    )
    print("FLAG 12 ✅ GRPOTrainer initialized")
except Exception as e:
    print(f"FLAG 12 ❌ {e}"); raise

print("\n" + "="*50)
print("ALL FLAGS PASSED — starting GRPO v4 training")
print("="*50)
print(f"  Dataset       : {len(hf_dataset)} multi-step pairs (fixed)")
print(f"  LR            : 1e-4")
print(f"  Group size G  : 256")
print(f"  KL beta       : 0.01")
print()
trainer.train()


FLAG 9 ✅ GRPOConfig created
FLAG 10 ✅ Trainable params cast to float32: 0
FLAG 11 ✅ GRPO LoRA config created
         Structure: Base → frozen SFT LoRA → trainable GRPO LoRA


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


FLAG 12 ✅ GRPOTrainer initialized

ALL FLAGS PASSED — starting GRPO v4 training
  Dataset       : 982 multi-step pairs (fixed)
  LR            : 1e-4
  Group size G  : 256
  KL beta       : 0.01



Step,Training Loss
10,0.000005
20,0.000014
30,0.000021
40,0.000027
50,0.000028
60,0.000029
70,0.000024
80,0.000029
90,0.000030


TrainOutput(global_step=90, training_loss=2.310462296009064e-05, metrics={'train_runtime': 3890.5666, 'train_samples_per_second': 0.757, 'train_steps_per_second': 0.023, 'total_flos': 0.0, 'train_loss': 2.310462296009064e-05})

In [ ]:
print("Saving GRPO v4...")
trainer.save_model(GRPO_CKPT)
tokenizer.save_pretrained(GRPO_CKPT)

cfg_path    = f"{GRPO_CKPT}/adapter_config.json"
weight_path = f"{GRPO_CKPT}/adapter_model.safetensors"
assert os.path.exists(cfg_path),    "❌ config missing"
assert os.path.exists(weight_path), "❌ weights missing"

with open(cfg_path) as f: cfg = json.load(f)
size_mb = os.path.getsize(weight_path) / 1e6
assert cfg.get('r') is not None, "❌ null config"
assert size_mb > 10,             "❌ weights too small"

print(f"✅ Saved: {size_mb:.1f} MB")
del trainer, model
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Saving GRPO v4...
✅ Saved: 73.9 MB
VRAM free: 73.5 GB


In [ ]:
import os, shutil, json
from google.colab import files

os.makedirs('./report_package_v4', exist_ok=True)

# ── 1. Results ──
if os.path.exists('./results'):
    shutil.copytree('./results', './report_package_v4/results',
                    dirs_exist_ok=True)
    print("Results copied")

# ── 2. Training logs ──
os.makedirs('./report_package_v4/logs', exist_ok=True)
log_files = [
    './checkpoints/grpo_qwen_textworld_v4/training_loss.png',
    './checkpoints/grpo_qwen_textworld_v4/trainer_log.jsonl',
    './checkpoints/grpo_qwen_textworld_v4/train_results.json',
]
for f in log_files:
    if os.path.exists(f):
        name = f.split('/')[-3] + '_' + f.split('/')[-1]
        shutil.copy(f, f'./report_package_v4/logs/{name}')
        print(f"Copied: {name}")

# ── 3. Essential checkpoint files only (skip large intermediate checkpoints) ──
essential = [
    'adapter_config.json',
    'adapter_model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json',
]
for method in ['grpo_qwen_textworld_v4']:
    ckpt_dir = f'./checkpoints/{method}'
    out_dir  = f'./report_package_v4/checkpoints/{method}'
    os.makedirs(out_dir, exist_ok=True)
    for f in essential:
        src = f'{ckpt_dir}/{f}'
        if os.path.exists(src):
            shutil.copy(src, f'{out_dir}/{f}')
            print(f"Copied: {method}/{f}")

# ── 4. Print what we have ──
print('\n=== Report Package Contents ===')
for root, dirs, fls in os.walk('./report_package_v4'):
    depth = root.replace('./report_package_v4', '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    for f in fls:
        size = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f'{indent}  {f}  ({size:.1f} MB)')

# ── 5. Zip and download ──
shutil.make_archive('./report_package_v4', 'zip', './report_package_v4')
size_mb = os.path.getsize('./report_package_v4.zip') / 1e6
print(f'\nZipping... {size_mb:.1f} MB total')
files.download('./report_package_v4.zip')
print('Download started.')

Results copied
Copied: grpo_qwen_textworld_v4/adapter_config.json
Copied: grpo_qwen_textworld_v4/adapter_model.safetensors
Copied: grpo_qwen_textworld_v4/tokenizer.json
Copied: grpo_qwen_textworld_v4/tokenizer_config.json

=== Report Package Contents ===
report_package_v4/
  results/
    ppo_results.json  (0.0 MB)
    all_results.json  (0.0 MB)
    bc_results.json  (0.0 MB)
    all_results.csv  (0.0 MB)
    grpo_results.json  (0.0 MB)
    ablation_sparse_rewards.json  (0.0 MB)
  checkpoints/
    grpo_qwen_textworld_v4/
      adapter_config.json  (0.0 MB)
      adapter_model.safetensors  (73.9 MB)
      tokenizer.json  (11.4 MB)
      tokenizer_config.json  (0.0 MB)
  logs/

Zipping... 70.4 MB total


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.


In [ ]:
import os, json, torch, textworld, gc, csv
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm
from google.colab import files

MODEL_BASE = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_CKPT   = "./checkpoints/sft_qwen_textworld"
GRPO_CKPT  = "./checkpoints/grpo_qwen_textworld_v4"
MAX_STEPS  = 50
BATCH_SIZE = 16

print("Loading GRPO model in bfloat16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base, SFT_CKPT, adapter_name="sft")
model.load_adapter(GRPO_CKPT, adapter_name="grpo")
model.set_adapter("grpo")
model.eval()

free = torch.cuda.mem_get_info()[0] / 1e9
print(f"VRAM free after model load: {free:.1f} GB")


def clean_action(raw, admissible):
    action = raw.split("\n")[0] \
                .split("Human:")[0] \
                .split("Assistant:")[0] \
                .split("Question:")[0] \
                .split("The best")[0]

    action = action.strip("()").strip(".").strip()

    adm_lower = [a.lower() for a in admissible]

    if action.lower() not in adm_lower:
        for adm in admissible:
            if adm.lower() in action.lower() or action.lower() in adm.lower():
                action = adm
                break

    return action


def get_actions_batched(
    observations,
    admissibles,
    histories,
    game_ids,
    verbose_flags=None
):
    prompts = []

    if verbose_flags is None:
        verbose_flags = [False] * len(observations)

    for local_i, (obs, adm) in enumerate(zip(observations, admissibles)):
        game_id = game_ids[local_i]
        recent = histories[game_id][-4:]
        recent_text = ", ".join(recent) if recent else "None"

        prompt = (
            "You are playing a text-based game.\n"
            "Choose the best action that makes progress toward winning.\n\n"

            f"Observation:\n{obs.strip()}\n\n"
            f"Recent actions:\n{recent_text}\n\n"
            f"Valid actions:\n{', '.join(adm[:10])}\n\n"

            "Rules:\n"
            "- Do not repeat actions that did not change the score.\n"
            "- Do not repeatedly examine the same object.\n"
            "- Prefer progress actions: go, take, open, unlock, put, insert, use, eat.\n"
            "- Reply with only one valid action from the list.\n\n"

            "Best action:"
        )

        msgs = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(text)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    actions = []
    input_len = inputs["input_ids"].shape[1]

    for local_i, out in enumerate(outputs):
        raw = tokenizer.decode(
            out[input_len:],
            skip_special_tokens=True
        ).strip()

        action = clean_action(raw, admissibles[local_i])
        is_valid = action.lower() in [a.lower() for a in admissibles[local_i]]

        if verbose_flags[local_i]:
            print(
                f"  [Game {game_ids[local_i]+1}] "
                f"raw='{raw[:60]}' → cleaned='{action}' → valid={is_valid}"
            )

        actions.append(action)

    return actions


def choose_fallback_action(admissible, history):
    bad_prefixes = ["examine", "look"]

    preferred_prefixes = [
        "take", "open", "unlock", "go", "put", "insert", "use", "eat"
    ]

    recent = set(a.lower() for a in history[-4:])

    for pref in preferred_prefixes:
        for adm in admissible:
            if adm.lower().startswith(pref) and adm.lower() not in recent:
                return adm

    for adm in admissible:
        if adm.lower() not in recent and not any(adm.lower().startswith(p) for p in bad_prefixes):
            return adm

    for adm in admissible:
        if adm.lower() not in recent:
            return adm

    return admissible[0]


def evaluate_batched(game_paths, desc="Evaluating", verbose_games=3):
    infos = textworld.EnvInfos(
        description=True,
        admissible_commands=True,
        score=True,
        won=True
    )

    envs = [textworld.start(gp, request_infos=infos) for gp in game_paths]
    states = [env.reset() for env in envs]

    total_scores = [0.0] * len(game_paths)
    steps_taken  = [0] * len(game_paths)
    won_flags    = [False] * len(game_paths)
    done_flags   = [False] * len(game_paths)

    histories = [[] for _ in game_paths]
    last_actions = [""] * len(game_paths)
    repeat_counts = [0] * len(game_paths)

    for step in tqdm(range(MAX_STEPS), desc=desc):
        active_idx = [i for i, d in enumerate(done_flags) if not d]
        if not active_idx:
            break

        for batch_start in range(0, len(active_idx), BATCH_SIZE):
            batch_idx = active_idx[batch_start:batch_start + BATCH_SIZE]

            observations = [states[i].description for i in batch_idx]
            admissibles  = [states[i].admissible_commands for i in batch_idx]
            verbose_flags = [i < verbose_games for i in batch_idx]

            actions = get_actions_batched(
                observations=observations,
                admissibles=admissibles,
                histories=histories,
                game_ids=batch_idx,
                verbose_flags=verbose_flags
            )

            for local_j, (i, action) in enumerate(zip(batch_idx, actions)):
                adm = admissibles[local_j]
                adm_lower = [a.lower() for a in adm]

                if action.lower() not in adm_lower:
                    action = choose_fallback_action(adm, histories[i])

                if repeat_counts[i] >= 2:
                    action = choose_fallback_action(adm, histories[i])
                    if i < verbose_games:
                        print(f"  [Game {i+1}] anti-loop fallback → '{action}'")

                states[i], score, done = envs[i].step(action)

                total_scores[i] += score
                steps_taken[i] += 1
                histories[i].append(action)

                if action.lower() == last_actions[i].lower() and score == 0:
                    repeat_counts[i] += 1
                else:
                    repeat_counts[i] = 0

                last_actions[i] = action

                if i < verbose_games:
                    print(
                        f"  [Game {i+1}] → executed='{action}' "
                        f"score={score} won={states[i].won}"
                    )

                if states[i].won:
                    won_flags[i] = True
                    done_flags[i] = True
                    if i < verbose_games:
                        print(f"  [Game {i+1}] WON in {steps_taken[i]} steps!")
                elif done:
                    done_flags[i] = True

    for env in envs:
        env.close()

    n = len(game_paths)
    return {
        "completion_rate": sum(won_flags) / n * 100,
        "avg_steps": sum(steps_taken) / n,
        "avg_score": sum(total_scores) / n,
        "n_games": n,
        "raw": [
            {"won": w, "steps": s, "total_score": t}
            for w, s, t in zip(won_flags, steps_taken, total_scores)
        ]
    }


print("\nEvaluating GRPO v4 on DENSE games...")
with open("./data/test_game_paths.json") as f:
    dense_paths = json.load(f)

grpo_dense = evaluate_batched(dense_paths, desc="GRPO Dense")
print(
    f"GRPO Dense  : {grpo_dense['completion_rate']:.1f}% | "
    f"Avg steps: {grpo_dense['avg_steps']:.1f} | "
    f"Avg score: {grpo_dense['avg_score']:.2f}"
)


print("\nEvaluating GRPO v4 on SPARSE games...")
with open("./data/test_sparse_game_paths.json") as f:
    sparse_paths = json.load(f)

grpo_sparse = evaluate_batched(sparse_paths, desc="GRPO Sparse")
print(
    f"GRPO Sparse : {grpo_sparse['completion_rate']:.1f}% | "
    f"Avg steps: {grpo_sparse['avg_steps']:.1f} | "
    f"Avg score: {grpo_sparse['avg_score']:.2f}"
)


with open("./results/all_results.json") as f:
    all_results = json.load(f)

all_results["GRPO"] = {
    k: v for k, v in grpo_dense.items() if k != "raw"
}

with open("./results/all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("✅ all_results.json updated")


with open("./results/ablation_sparse_rewards.json") as f:
    sparse_results = json.load(f)

sparse_results["GRPO"] = {
    k: v for k, v in grpo_sparse.items() if k != "raw"
}

with open("./results/ablation_sparse_rewards.json", "w") as f:
    json.dump(sparse_results, f, indent=2)

print("✅ ablation_sparse_rewards.json updated")


print("\n" + "=" * 58)
print("TABLE 1: Main Comparison (Dense Rewards)")
print("=" * 58)
print(f'{"Method":<10} {"Completion %":>13} {"Avg Steps":>10} {"Avg Score":>10}')
print("-" * 58)

for m in ["BC", "PPO", "GRPO"]:
    if m in all_results:
        r = all_results[m]
        print(
            f"{m:<10} {r['completion_rate']:>12.1f}% "
            f"{r['avg_steps']:>10.1f} {r['avg_score']:>10.2f}"
        )


print("\n" + "=" * 58)
print("TABLE 2: Dense vs Sparse")
print("=" * 58)
print(f'{"Method":<10} {"Dense %":>8} {"Sparse %":>9} {"Drop":>8}')
print("-" * 58)

for m in ["BC", "PPO", "GRPO"]:
    d = all_results.get(m, {}).get("completion_rate")
    s = sparse_results.get(m, {}).get("completion_rate")
    if d is not None and s is not None:
        print(f"{m:<10} {d:>7.1f}% {s:>8.1f}% {d-s:>7.1f}%")


rows = []

for m in ["BC", "PPO", "GRPO"]:
    if m in all_results:
        r = all_results[m]
        rows.append({
            "experiment": "main",
            "method": m,
            "reward": "dense",
            "completion_rate": round(r["completion_rate"], 2),
            "avg_steps": round(r["avg_steps"], 2),
            "avg_score": round(r["avg_score"], 2)
        })

for m in ["BC", "PPO", "GRPO"]:
    if m in sparse_results:
        r = sparse_results[m]
        rows.append({
            "experiment": "sparse",
            "method": m,
            "reward": "sparse",
            "completion_rate": round(r["completion_rate"], 2),
            "avg_steps": round(r["avg_steps"], 2),
            "avg_score": round(r["avg_score"], 2)
        })

csv_path = "./results/all_results.csv"

with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print("✅ all_results.csv updated")
files.download(csv_path)
print("✅ CSV downloaded.")

Loading GRPO model in bfloat16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM free after model load: 73.0 GB

Evaluating GRPO v4 on DENSE games...


GRPO Dense:   0%|          | 0/50 [00:00<?, ?it/s]

  [Game 1] raw='go south' → cleaned='go south' → valid=True
  [Game 2] raw='go east' → cleaned='go east' → valid=True
  [Game 3] raw='go east' → cleaned='go east' → valid=True
  [Game 1] → executed='go south' score=0 won=False
  [Game 2] → executed='go east' score=0 won=False
  [Game 3] → executed='go east' score=0 won=False


...

GRPO Sparse: 100%|██████████| 50/50 [03:38<00:00,  4.37s/it]

GRPO Sparse : 10.0% | Avg steps: 46.0 | Avg score: 0.10
✅ all_results.json updated
✅ ablation_sparse_rewards.json updated

TABLE 1: Main Comparison (Dense Rewards)
Method      Completion %  Avg Steps  Avg Score
----------------------------------------------------------
BC                 12.0%       44.6       0.12
PPO                22.0%       40.1       0.22
GRPO               10.0%       46.0       0.10

TABLE 2: Dense vs Sparse
Method      Dense %  Sparse %     Drop
----------------------------------------------------------
BC            12.0%     12.0%     0.0%
PPO           22.0%     22.0%     0.0%
GRPO          10.0%     10.0%     0.0%
✅ all_results.csv updated


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ CSV downloaded.
